# Synthetic Time-Series Generation with a TSGM Variational Autoencoder

This notebook trains a **Variational Autoencoder (VAE)** using the
[TSGM](https://github.com/AlexanderVNikitin/tsgm) (Time Series Generative
Modeling) library to generate synthetic samples for any of the
preprocessed multimodal time-series datasets in `datasets/`:

- `X_imu_raw.npy`
- `X_imu_ir_raw.npy`
- `X_imu_radar_raw.npy`
- `X_imu_ir_radar_raw.npy`
- `metadata.csv`

**The only cell you need to edit is Section 2 (Configuration).** Everything
else automatically adapts to the dataset you select (sequence length,
number of features, output folders, etc.).

The notebook is organised as follows:

1. Imports
2. Configuration
3. Dataset Selection
4. Data Validation
5. Train / Test Split (+ scaling)
6. Build the VAE
7. Training
8. Generate Synthetic Samples
9. Assign Metadata
10. Save Results
11. Visualization
12. Quality Evaluation
13. Reproducibility


## Section 1 — Imports

We import everything the notebook needs. If a package is missing (TSGM,
TensorFlow, scikit-learn, ...) it is installed automatically with `pip`,
so the notebook can run end-to-end with minimal setup.

**Important:** the Keras backend must be selected *before* TSGM/Keras are
imported for the first time, so we set `KERAS_BACKEND=tensorflow` first.


In [ ]:
import os
import sys
import subprocess
import warnings

# TSGM (via Keras 3) supports several backends (tensorflow / torch / jax).
# We pin TensorFlow explicitly *before* importing tsgm/keras so the backend
# selection is deterministic regardless of what else is installed.
os.environ.setdefault("KERAS_BACKEND", "tensorflow")


def _ensure_package(pip_name: str, import_name: str = None) -> None:
    """Install `pip_name` via pip if `import_name` cannot be imported."""
    import_name = import_name or pip_name
    try:
        __import__(import_name)
    except ImportError:
        print(f"Installing missing package: {pip_name} ...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet",
             "--break-system-packages", pip_name]
        )


for _pip_name, _import_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("scikit-learn", "sklearn"),
    ("tensorflow-cpu", "tensorflow"),
    ("tsgm", "tsgm"),
]:
    try:
        _ensure_package(_pip_name, _import_name)
    except Exception as exc:  # pragma: no cover - best-effort auto-install
        print(f"WARNING: could not auto-install '{_pip_name}': {exc}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.svm import OneClassSVM

import tensorflow as tf
import keras
from keras import ops

import tsgm
import tsgm.models
from tsgm.models.architectures import zoo as tsgm_zoo

warnings.filterwarnings("ignore")

print(f"TensorFlow version : {tf.__version__}")
print(f"Keras version       : {keras.__version__}")
print(f"Keras backend       : {keras.backend.backend()}")
print(f"TSGM version        : {getattr(tsgm, '__version__', 'unknown')}")

## Section 2 — Configuration

**This is the only cell you should need to edit.** Change `DATASET_NAME`
to switch between the four datasets — nothing else in the notebook needs
to be touched.


In [ ]:
# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
DATASET_DIR = "./datasets"          # folder containing the .npy files + metadata.csv
DATASET_NAME = "X_imu_raw"          # one of: X_imu_raw, X_imu_ir_raw, X_imu_radar_raw, X_imu_ir_radar_raw
METADATA_FILE = "metadata.csv"      # metadata file name, inside DATASET_DIR
OUTPUT_DIR = "./vae_generated"      # root folder for all generated artifacts

# ---------------------------------------------------------------------
# VAE / training hyperparameters
# ---------------------------------------------------------------------
LATENT_DIM = 32
EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 0.001
BETA = 1.0                          # beta-VAE weight on the KL term
VAL_SPLIT = 0.2                     # fraction of samples used for validation

# ---------------------------------------------------------------------
# Generation
# ---------------------------------------------------------------------
N_SYNTHETIC_SAMPLES = 1000

# ---------------------------------------------------------------------
# Metadata assignment strategy: "random" or "balanced"
#   - "random"   -> randomly sample metadata rows from the original data
#   - "balanced" -> generate a class-balanced set of metadata rows, based
#                   on LABEL_COLUMN (only used if that column exists)
# ---------------------------------------------------------------------
METADATA_STRATEGY = "random"
LABEL_COLUMN = "label"              # set to None to disable label-aware logic

# ---------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------
RANDOM_SEED = 42

print("Configuration loaded:")
for _name in [
    "DATASET_DIR", "DATASET_NAME", "METADATA_FILE", "OUTPUT_DIR",
    "LATENT_DIM", "EPOCHS", "BATCH_SIZE", "LEARNING_RATE", "BETA", "VAL_SPLIT",
    "N_SYNTHETIC_SAMPLES", "METADATA_STRATEGY", "LABEL_COLUMN", "RANDOM_SEED",
]:
    print(f"  {_name:20s} = {globals()[_name]!r}")

## Section 3 — Dataset Selection

The dataset path is built automatically from `DATASET_DIR` and
`DATASET_NAME`. `metadata.csv` is loaded from the same folder. This cell
does not need editing for a different dataset — just change
`DATASET_NAME` in Section 2.


In [ ]:
dataset_path = Path(DATASET_DIR) / f"{DATASET_NAME}.npy"
metadata_path = Path(DATASET_DIR) / METADATA_FILE

if not dataset_path.exists():
    raise FileNotFoundError(
        f"Could not find dataset file '{dataset_path}'. "
        f"Check DATASET_DIR/DATASET_NAME in the configuration cell."
    )
if not metadata_path.exists():
    raise FileNotFoundError(
        f"Could not find metadata file '{metadata_path}'. "
        f"Check DATASET_DIR/METADATA_FILE in the configuration cell."
    )

X = np.load(dataset_path).astype("float32")
metadata = pd.read_csv(metadata_path)

if X.ndim != 3:
    raise ValueError(
        f"Expected a 3D array (n_samples, seq_len, n_features), got shape {X.shape}."
    )

N_SAMPLES, SEQ_LEN, N_FEATURES = X.shape

print(f"Dataset selected     : {DATASET_NAME}")
print(f"Data shape            : {X.shape}")
print(f"Number of samples     : {N_SAMPLES}")
print(f"Sequence length       : {SEQ_LEN}")
print(f"Number of features    : {N_FEATURES}")
print(f"Metadata shape        : {metadata.shape}")
metadata.head()

## Section 4 — Data Validation

Basic sanity checks before training: no NaNs, no infinities, correct
number of dimensions, and metadata/feature alignment. We also print
per-feature summary statistics to get a feel for the data's scale.


In [ ]:
n_nan = np.isnan(X).sum()
n_inf = np.isinf(X).sum()

print("=== Data validation ===")
print(f"NaN values found       : {n_nan}")
print(f"Infinite values found  : {n_inf}")
print(f"Array ndim             : {X.ndim} (expected 3)")

if n_nan > 0:
    raise ValueError(f"Dataset '{DATASET_NAME}' contains {n_nan} NaN values. Please clean the data first.")
if n_inf > 0:
    raise ValueError(f"Dataset '{DATASET_NAME}' contains {n_inf} infinite values. Please clean the data first.")
if X.ndim != 3:
    raise ValueError(f"Expected a 3D array, got ndim={X.ndim}.")

if len(metadata) != N_SAMPLES:
    print(
        f"WARNING: metadata has {len(metadata)} rows but X has {N_SAMPLES} samples. "
        f"Metadata-related steps (Sections 9-10) will sample with replacement as needed."
    )

print("\n=== Per-feature statistics (flattened over samples & time) ===")
flat = X.reshape(-1, N_FEATURES)
stats = pd.DataFrame({
    "min": flat.min(axis=0),
    "max": flat.max(axis=0),
    "mean": flat.mean(axis=0),
    "std": flat.std(axis=0),
})
print(stats.describe().loc[["mean", "min", "max"]])
print("\nValidation passed: dataset is clean and well-formed.")

## Section 5 — Train / Validation Split

We split the samples (and their aligned metadata rows, when the row
counts match) into training and validation sets. Because the VAE decoder
uses a `sigmoid` output activation (values in `[0, 1]`), we additionally
fit a `MinMaxScaler` **on the training data only** and apply it to both
splits, to avoid data leakage. The scaler is kept so generated samples
can be mapped back to the original scale later (Section 8).


In [ ]:
indices = np.arange(N_SAMPLES)
train_idx, val_idx = train_test_split(
    indices, test_size=VAL_SPLIT, random_state=RANDOM_SEED
)

X_train_raw, X_val_raw = X[train_idx], X[val_idx]

# Fit the scaler on the training data only (per-feature), then transform both splits.
scaler = MinMaxScaler(feature_range=(0.0, 1.0))
n_train = X_train_raw.shape[0]
scaler.fit(X_train_raw.reshape(-1, N_FEATURES))

def scale(arr):
    flat = arr.reshape(-1, N_FEATURES)
    return scaler.transform(flat).reshape(arr.shape).astype("float32")

def unscale(arr):
    flat = arr.reshape(-1, N_FEATURES)
    return scaler.inverse_transform(flat).reshape(arr.shape).astype("float32")

X_train = scale(X_train_raw)
X_val = scale(X_val_raw)

print(f"Train samples : {X_train.shape[0]}")
print(f"Val samples   : {X_val.shape[0]}")
print(f"Train range after scaling : [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"Val range after scaling   : [{X_val.min():.3f}, {X_val.max():.3f}]")

## Section 6 — Build the VAE

We use TSGM's ready-to-use convolutional VAE architecture
(`tsgm.models.architectures.zoo["vae_conv5"]`), which provides a
clearly-defined encoder/decoder pair for time series, and wrap it with
TSGM's `BetaVAE` model class (`tsgm.models.cvae.BetaVAE`), which handles
the reparameterization trick and the reconstruction + KL loss for us.
The latent dimensionality is fully configurable via `LATENT_DIM`.


In [ ]:
architecture = tsgm_zoo["vae_conv5"](
    seq_len=SEQ_LEN,
    feat_dim=N_FEATURES,
    latent_dim=LATENT_DIM,
)
encoder = architecture.encoder
decoder = architecture.decoder

print("=== Encoder ===")
encoder.summary()
print("\n=== Decoder ===")
decoder.summary()

vae = tsgm.models.cvae.BetaVAE(encoder, decoder, beta=BETA)
vae.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE))
print("\nVAE built and compiled successfully.")

## Section 7 — Training

TSGM's `BetaVAE` implements a custom `train_step` but no `test_step`, so
Keras' built-in `validation_data` mechanism for `.fit()` does not apply
here directly. Instead, we use a small callback that recomputes the same
reconstruction + KL loss on the validation set at the end of every epoch,
so we can track and plot both training and validation curves.


In [ ]:
def _vae_loss_components(model, data):
    """Recompute (total, reconstruction, kl) loss for a batch of data."""
    z_mean, z_log_var, z = model.encoder(data)
    reconstruction = model.decoder(z_mean)
    reconstruction_loss = float(np.mean(np.array(model._get_reconstruction_loss(data, reconstruction))))
    kl = -0.5 * (1 + z_log_var - ops.square(z_mean) - ops.exp(z_log_var))
    kl_loss = float(np.array(ops.mean(ops.sum(kl, axis=1))))
    total_loss = reconstruction_loss + float(model.beta) * kl_loss
    return total_loss, reconstruction_loss, kl_loss


class ValidationLossCallback(keras.callbacks.Callback):
    """Tracks validation loss/reconstruction/KL at the end of each epoch."""

    def __init__(self, X_val):
        super().__init__()
        self.X_val = X_val
        self.history = {"val_loss": [], "val_reconstruction_loss": [], "val_kl_loss": []}

    def on_epoch_end(self, epoch, logs=None):
        total, recon, kl = _vae_loss_components(self.model, self.X_val)
        self.history["val_loss"].append(total)
        self.history["val_reconstruction_loss"].append(recon)
        self.history["val_kl_loss"].append(kl)
        print(f"  -> val_loss: {total:.4f} - val_reconstruction_loss: {recon:.4f} - val_kl_loss: {kl:.4f}")


val_callback = ValidationLossCallback(X_val)

history = vae.fit(
    X_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    verbose=2,
    callbacks=[val_callback],
)

train_hist = history.history
val_hist = val_callback.history

print("\nTraining complete.")
print(f"Final train loss : {train_hist['loss'][-1]:.4f}")
print(f"Final val loss    : {val_hist['val_loss'][-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, key, title in zip(
    axes,
    ["loss", "reconstruction_loss", "kl_loss"],
    ["Total loss", "Reconstruction loss", "KL loss"],
):
    ax.plot(train_hist[key], label="train")
    ax.plot(val_hist[f"val_{key}"], label="validation")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(alpha=0.3)

fig.suptitle(f"VAE training curves — {DATASET_NAME}")
fig.tight_layout()
plt.show()

## Section 8 — Generate Synthetic Samples

We sample `N_SYNTHETIC_SAMPLES` new sequences from the trained VAE by
decoding random draws from the latent prior. The output is then mapped
back to the original data scale (inverse of the `MinMaxScaler` fit in
Section 5), so the generated data has exactly the same dimensions *and*
the same value range as the original dataset.


In [ ]:
generated_scaled = np.array(vae.generate(N_SYNTHETIC_SAMPLES))
synthetic_X = unscale(generated_scaled)

print(f"Original dataset shape : {X.shape}")
print(f"Generated data shape   : {synthetic_X.shape}")

assert synthetic_X.shape[1:] == X.shape[1:], (
    "Generated data does not match the original feature dimensions!"
)
print("\nShape check passed: generated samples match the original (seq_len, n_features).")

## Section 9 — Assign Metadata

Since the VAE only generates feature sequences, we need to attach
metadata rows to the synthetic samples. Two strategies are supported,
controlled by `METADATA_STRATEGY` in the configuration cell:

- **`"random"`** — randomly sample (with replacement) rows from the
  original metadata.
- **`"balanced"`** — if `LABEL_COLUMN` exists in the metadata, generate a
  class-balanced set of metadata rows (roughly equal count per class),
  drawing real rows from each class to populate them.

Every synthetic row also gets a fresh identifier and a `source` column
so real and synthetic samples can be told apart later.


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

def _sample_random_metadata(n):
    sampled_idx = rng.integers(0, len(metadata), size=n)
    return metadata.iloc[sampled_idx].reset_index(drop=True)


def _sample_balanced_metadata(n, label_column):
    classes = metadata[label_column].unique()
    n_classes = len(classes)
    base, remainder = divmod(n, n_classes)
    counts = {cls: base for cls in classes}
    # Distribute the remainder across the first few classes
    for cls in list(classes)[:remainder]:
        counts[cls] += 1

    rows = []
    for cls, count in counts.items():
        class_rows = metadata[metadata[label_column] == cls]
        sampled_idx = rng.integers(0, len(class_rows), size=count)
        rows.append(class_rows.iloc[sampled_idx])
    return pd.concat(rows, ignore_index=True).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)


use_balanced = (
    METADATA_STRATEGY == "balanced"
    and LABEL_COLUMN is not None
    and LABEL_COLUMN in metadata.columns
)

if METADATA_STRATEGY == "balanced" and not use_balanced:
    print(
        f"WARNING: METADATA_STRATEGY='balanced' requested but LABEL_COLUMN "
        f"'{LABEL_COLUMN}' was not found in metadata columns {list(metadata.columns)}. "
        f"Falling back to 'random'."
    )

if use_balanced:
    synthetic_metadata = _sample_balanced_metadata(N_SYNTHETIC_SAMPLES, LABEL_COLUMN)
    print(f"Assigned class-balanced metadata using label column '{LABEL_COLUMN}'.")
else:
    synthetic_metadata = _sample_random_metadata(N_SYNTHETIC_SAMPLES)
    print("Assigned metadata via random sampling from the original metadata.")

synthetic_metadata = synthetic_metadata.drop(columns=["sample_id", "source"], errors="ignore")
synthetic_metadata.insert(0, "sample_id", [f"synthetic_{i:05d}" for i in range(N_SYNTHETIC_SAMPLES)])
synthetic_metadata["source"] = "synthetic"

print(f"Synthetic metadata shape: {synthetic_metadata.shape}")
synthetic_metadata.head()

## Section 10 — Save Results

All artifacts are written to a dataset-specific subfolder of
`OUTPUT_DIR`, created automatically if it doesn't exist:

```
vae_generated/
    <DATASET_NAME>/
        synthetic_X.npy
        synthetic_metadata.csv
        combined_X.npy
        combined_metadata.csv
```

`combined_*` files contain the original data with the synthetic data
appended, ready to be used for downstream augmentation experiments.


In [ ]:
output_dir = Path(OUTPUT_DIR) / DATASET_NAME
output_dir.mkdir(parents=True, exist_ok=True)

original_metadata = metadata.copy()
if "sample_id" not in original_metadata.columns:
    original_metadata.insert(0, "sample_id", [f"real_{i:05d}" for i in range(len(original_metadata))])
original_metadata["source"] = "real"

combined_X = np.concatenate([X, synthetic_X], axis=0)
combined_metadata = pd.concat([original_metadata, synthetic_metadata], ignore_index=True)

np.save(output_dir / "synthetic_X.npy", synthetic_X)
synthetic_metadata.to_csv(output_dir / "synthetic_metadata.csv", index=False)

np.save(output_dir / "combined_X.npy", combined_X)
combined_metadata.to_csv(output_dir / "combined_metadata.csv", index=False)

print(f"Saved outputs to: {output_dir.resolve()}")
for f in sorted(output_dir.iterdir()):
    print(f"  - {f.name}")

## Section 11 — Visualization

We compare real and synthetic data visually:

- Real vs. synthetic time series for a few random samples/features
- Feature-wise distributions (real vs. synthetic)
- PCA projection (2D)
- t-SNE projection (2D)


In [ ]:
n_show = min(3, N_FEATURES)
feature_idx_to_show = rng.choice(N_FEATURES, size=n_show, replace=False)

fig, axes = plt.subplots(n_show, 1, figsize=(12, 3 * n_show), squeeze=False)
real_sample = X[rng.integers(0, N_SAMPLES)]
synth_sample = synthetic_X[rng.integers(0, N_SYNTHETIC_SAMPLES)]

for row, feat in enumerate(feature_idx_to_show):
    ax = axes[row][0]
    ax.plot(real_sample[:, feat], label="real", linewidth=2)
    ax.plot(synth_sample[:, feat], label="synthetic", linewidth=2, linestyle="--")
    ax.set_title(f"Feature {feat}")
    ax.set_xlabel("Time step")
    ax.legend()
    ax.grid(alpha=0.3)

fig.suptitle(f"Real vs. synthetic time series — {DATASET_NAME}")
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 4), squeeze=False)
real_flat = X.reshape(-1, N_FEATURES)
synth_flat = synthetic_X.reshape(-1, N_FEATURES)

for col, feat in enumerate(feature_idx_to_show):
    ax = axes[0][col]
    ax.hist(real_flat[:, feat], bins=30, alpha=0.5, label="real", density=True)
    ax.hist(synth_flat[:, feat], bins=30, alpha=0.5, label="synthetic", density=True)
    ax.set_title(f"Feature {feat} distribution")
    ax.legend()
    ax.grid(alpha=0.3)

fig.suptitle(f"Feature distributions: real vs. synthetic — {DATASET_NAME}")
fig.tight_layout()
plt.show()

In [ ]:
real_2d = X.reshape(N_SAMPLES, -1)
synth_2d = synthetic_X.reshape(N_SYNTHETIC_SAMPLES, -1)
combined_2d = np.concatenate([real_2d, synth_2d], axis=0)
labels_2d = np.array(["real"] * N_SAMPLES + ["synthetic"] * N_SYNTHETIC_SAMPLES)

pca = PCA(n_components=2, random_state=RANDOM_SEED)
pca_proj = pca.fit_transform(combined_2d)

fig, ax = plt.subplots(figsize=(6, 5))
for label in ["real", "synthetic"]:
    mask = labels_2d == label
    ax.scatter(pca_proj[mask, 0], pca_proj[mask, 1], alpha=0.6, label=label, s=25)
ax.set_title(f"PCA projection — {DATASET_NAME}")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# t-SNE can be slow / unstable on very small sample counts; guard the perplexity.
n_total = combined_2d.shape[0]
perplexity = max(5, min(30, n_total // 4))

try:
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=RANDOM_SEED, init="pca")
    tsne_proj = tsne.fit_transform(combined_2d)

    fig, ax = plt.subplots(figsize=(6, 5))
    for label in ["real", "synthetic"]:
        mask = labels_2d == label
        ax.scatter(tsne_proj[mask, 0], tsne_proj[mask, 1], alpha=0.6, label=label, s=25)
    ax.set_title(f"t-SNE projection — {DATASET_NAME}")
    ax.set_xlabel("Dim 1")
    ax.set_ylabel("Dim 2")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.show()
except Exception as exc:
    print(f"t-SNE visualization skipped: {exc}")

## Section 12 — Quality Evaluation

Simple, interpretable similarity metrics between real and synthetic data:

- Per-feature mean and variance (real vs. synthetic)
- MSE between randomly matched real/synthetic sample pairs
- Correlation matrices (real vs. synthetic)

Each metric is followed by a short, plain-language interpretation.


In [ ]:
real_mean, real_var = real_flat.mean(axis=0), real_flat.var(axis=0)
synth_mean, synth_var = synth_flat.mean(axis=0), synth_flat.var(axis=0)

mean_diff = np.abs(real_mean - synth_mean)
var_diff = np.abs(real_var - synth_var)

print("=== Mean / variance comparison (first 10 features) ===")
comparison = pd.DataFrame({
    "real_mean": real_mean[:10], "synthetic_mean": synth_mean[:10], "abs_mean_diff": mean_diff[:10],
    "real_var": real_var[:10], "synthetic_var": synth_var[:10], "abs_var_diff": var_diff[:10],
})
print(comparison)

print(
    f"\nAverage absolute mean difference across all features : {mean_diff.mean():.4f}\n"
    f"Average absolute variance difference across all features: {var_diff.mean():.4f}\n"
    "Interpretation: smaller values indicate the synthetic data preserves the "
    "first- and second-order statistics of the real data more closely."
)

In [ ]:
n_pairs = min(N_SAMPLES, N_SYNTHETIC_SAMPLES, 200)
real_pair_idx = rng.integers(0, N_SAMPLES, size=n_pairs)
synth_pair_idx = rng.integers(0, N_SYNTHETIC_SAMPLES, size=n_pairs)

mse_per_pair = np.mean(
    (X[real_pair_idx] - synthetic_X[synth_pair_idx]) ** 2, axis=(1, 2)
)

print("=== MSE between randomly matched real/synthetic samples ===")
print(f"Mean MSE : {mse_per_pair.mean():.4f}")
print(f"Std  MSE : {mse_per_pair.std():.4f}")
print(
    "Interpretation: this is a rough, non-aligned similarity score (samples are "
    "randomly paired, not matched by nearest neighbour), so it mainly indicates "
    "whether synthetic samples live on a similar numerical scale to real ones."
)

In [ ]:
real_corr = np.corrcoef(real_flat, rowvar=False)
synth_corr = np.corrcoef(synth_flat, rowvar=False)
corr_diff = np.abs(real_corr - synth_corr)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
n_feat_show = min(N_FEATURES, 50)  # cap for readability on high-dimensional datasets
im0 = axes[0].imshow(real_corr[:n_feat_show, :n_feat_show], cmap="coolwarm", vmin=-1, vmax=1)
axes[0].set_title("Real correlation")
im1 = axes[1].imshow(synth_corr[:n_feat_show, :n_feat_show], cmap="coolwarm", vmin=-1, vmax=1)
axes[1].set_title("Synthetic correlation")
im2 = axes[2].imshow(corr_diff[:n_feat_show, :n_feat_show], cmap="viridis")
axes[2].set_title("Absolute difference")
for ax, im in zip(axes, [im0, im1, im2]):
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle(f"Feature correlation structure — {DATASET_NAME} (showing first {n_feat_show} features)")
fig.tight_layout()
plt.show()

print(f"Mean absolute correlation difference: {np.nanmean(corr_diff):.4f}")
print(
    "Interpretation: values close to 0 mean the synthetic data preserves the "
    "linear relationships between features observed in the real data."
)

## Section 13 — Reproducibility

Random seeds were set at the very start of the notebook run (right after
loading the configuration) so that dataset splitting, weight
initialization, training, and sample generation are reproducible given
the same `RANDOM_SEED`. This cell re-applies the seeds explicitly for
clarity and can be re-run safely at any point.


In [ ]:
import random

def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_all_seeds(RANDOM_SEED)
print(f"Random seeds (python, numpy, tensorflow) set to {RANDOM_SEED} for reproducibility.")